# MNIST MLP3: AdamW versus Spectral RG-Flow Projector

This notebook tests a new optimizer experiment that is **not** a trace-log-normal projection.
WeightWatcher supplies the ESD, $\alpha$, and the PL boundary. The adaptive bulk-effective method
supplies the self-consistent ECS. On the resulting working support, the optimizer compares the
centered log-eigenvalue shapes before and after the completed AdamW step:

$$
z_i = \log s_i^2-\frac{1}{m}\sum_{j=1}^m \log s_j^2,
\qquad
\Delta z=z_{\mathrm{AdamW}}-z_{\mathrm{before}}.
$$

The RG draft identifies the trivial branch by loss of an extensive ECS, but it does not uniquely
derive a local differentiable vector toward that branch. This notebook therefore tests an explicit
surrogate: increasing retained spectral concentration, measured by

$$
C_{F_0}(z)=\log\sum_i p_i^2=-\log r_{\mathrm{PR}},
\qquad
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

Its centered spectral vector is

$$
v_{F_0,i}=2\left(\frac{p_i^2}{\sum_j p_j^2}-p_i\right),
\qquad \sum_i v_{F_0,i}=0.
$$

Only positive flow toward this collapse surrogate is removed:

$$
a_+ = \max\left(\frac{\langle\Delta z,v_{F_0}\rangle}{\lVert v_{F_0}\rVert^2},0\right),
\qquad
\Delta z_{\mathrm{RG}}=\Delta z-a_+v_{F_0}.
$$

The primary falsification test is whether FC1 remains closer to $\alpha=2$ than matched AdamW
without damaging test accuracy.


In [ ]:
# Optional WeightWatcher install. PyTorch/torchvision are assumed to match the local machine.
import importlib, subprocess, sys
try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "weightwatcher>=0.7.7"])


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
package_root = None
for root in [cwd, *cwd.parents]:
    direct = root / "rg_spectral_flow"
    nested = root / "optimizers" / "spectral_rg_flow_projector" / "rg_spectral_flow"
    if direct.is_dir():
        package_root = root
        break
    if nested.is_dir():
        package_root = nested.parent
        break
if package_root is None:
    raise RuntimeError("Run this notebook from a clone of CalculatedContent/rg_optimizers.")
sys.path.insert(0, str(package_root))
print("Using package root:", package_root)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rg_spectral_flow import MNISTExperimentConfig, run_mnist_comparison

pd.set_option("display.max_columns", None)
CONFIG = MNISTExperimentConfig(
    epochs=20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    collapse_potential="participation_ratio",
    projection_strength=1.0,
    max_abs_log_eigenvalue_correction=0.20,
    max_correction_ratio=0.10,
    apply_every_steps=25,
    warmup_epochs=1,
    min_retained=20,
    sc_effective_rank_method="participation_ratio",
    sc_normalization_gamma=0.0,
    sc_support_policy="midpoint",
    ww_svd_method="accurate",
)
CONFIG


In [ ]:
result = run_mnist_comparison(CONFIG, data_dir="./data", progress=True)
RUN_DIR = Path("./runs_mnist_spectral_rg_flow")
result.save(RUN_DIR)
print("Saved:", RUN_DIR.resolve())


## Checkpoint diagnostics

Monitor `alpha`, `ERG_gap_SC`, and test accuracy. The old full-$M$ gap is retained only as an audit.


In [ ]:
cols = [
    "run", "epoch", "layer_name", "alpha", "num_pl_spikes",
    "detX_num_WW", "ERG_gap_WW", "detX_num_SC", "ERG_gap_SC",
    "m_working", "M_normalization_SC", "SC_status",
]
display(result.weightwatcher[cols].tail(24))


In [ ]:
# Accuracy comparison.
fig, ax = plt.subplots(figsize=(9, 5))
for run, frame in result.performance.groupby("run"):
    frame = frame.sort_values("epoch")
    ax.plot(frame["epoch"], frame["test_acc"], marker="o", label=f"{run}: test")
    ax.plot(frame["epoch"], frame["train_acc"], linestyle="--", label=f"{run}: train")
ax.set(xlabel="Epoch", ylabel="Accuracy", title="AdamW versus Spectral RG-Flow Projector")
ax.grid(True, alpha=0.3); ax.legend(); plt.show()


In [ ]:
# FC1 is the primary test; also show the other layers.
valid = result.weightwatcher[result.weightwatcher["status"] == "ok"].copy()
for run, run_frame in valid.groupby("run"):
    fig, ax = plt.subplots(figsize=(9, 5))
    for layer, frame in run_frame.groupby("layer_name"):
        frame = frame.sort_values("epoch")
        ax.plot(frame["epoch"], frame["alpha"], marker="o", label=layer)
    ax.axhline(2.0, linestyle="--", label="alpha = 2")
    ax.set(xlabel="Epoch", ylabel="WeightWatcher alpha", title=f"Layer alpha: {run}")
    ax.grid(True, alpha=0.3); ax.legend(); plt.show()


In [ ]:
# Confirm that the measured component is removed when a correction fires.
if result.flow_steps.empty:
    print("No flow evaluations were logged.")
else:
    display(result.correction_summary.tail(20))
    applied = result.flow_steps[result.flow_steps["status"] == "ok"]
    fig, ax = plt.subplots(figsize=(9, 5))
    for parameter, frame in applied.groupby("parameter"):
        frame = frame.sort_values("global_step")
        ax.plot(frame["global_step"], frame["base_flow_component"], label=f"{parameter}: base")
        ax.plot(frame["global_step"], frame["corrected_flow_component"], linestyle="--", label=f"{parameter}: corrected")
    ax.axhline(0.0, linestyle=":")
    ax.set(xlabel="Global step", ylabel="Projection onto collapse vector", title="Removed spectral flow toward F0 surrogate")
    ax.grid(True, alpha=0.3); ax.legend(); plt.show()


In [ ]:
# Effective rank before/base/corrected: the one-sided projection should oppose retained-rank collapse.
if not result.correction_summary.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    for parameter, frame in result.correction_summary.groupby("parameter"):
        frame = frame.sort_values("epoch")
        ax.plot(frame["epoch"], frame["mean_effective_rank_base"], marker="o", label=f"{parameter}: AdamW proposal")
        ax.plot(frame["epoch"], frame["mean_effective_rank_corrected"], marker="s", label=f"{parameter}: corrected")
    ax.set(xlabel="Epoch", ylabel="Retained participation-ratio rank", title="Effect of the spectral flow subtraction")
    ax.grid(True, alpha=0.3); ax.legend(); plt.show()
